In [1]:
# Validação e Tratamento de Dados Ausentes

import urllib.request
import csv
import io
from tabulate import tabulate

# Baixar e ler o arquivo de produtos
def baixar_linhas_csv_da_url(url):
    with urllib.request.urlopen(url) as response:
        dados = response.read().decode('utf-8')
    return list(csv.reader(io.StringIO(dados)))

# Verificar se um campo está ausente
def campo_ausente(valor):
    valor = valor.strip()
    return valor == '' or valor.lower() == 'null'

# Converter string para float, retornando None se não for possível
def converter_para_float(valor):
    try:
        return float(valor)
    except:
        return None

# Calcular a média de uma coluna, ignorando valores ausentes
def calcular_media_coluna(dados, indice):
    soma = 0.0
    quantidade = 0
    for linha in dados:
        valor = linha[indice].strip()
        if not campo_ausente(valor):
            numero = converter_para_float(valor)
            if numero is not None:
                soma += numero
                quantidade += 1
    return soma / quantidade if quantidade > 0 else 0

# Corrigir uma linha, preenchendo campos ausentes com valores padrão ou médias
def corrigir_linha(linha, indices, medias):
    if campo_ausente(linha[indices['categoria']]):
        linha[indices['categoria']] = 'Sem categoria'
    if campo_ausente(linha[indices['peso']]):
        linha[indices['peso']] = str(round(medias['peso'], 2))
    if campo_ausente(linha[indices['comprimento']]):
        linha[indices['comprimento']] = str(round(medias['comprimento'], 2))
    if campo_ausente(linha[indices['altura']]):
        linha[indices['altura']] = str(round(medias['altura'], 2))
    if campo_ausente(linha[indices['largura']]):
        linha[indices['largura']] = str(round(medias['largura'], 2))
    return linha


def salvar_csv(caminho_arquivo, linhas):
    with open(caminho_arquivo, 'w', encoding='utf-8', newline='') as f_out:
        escritor = csv.writer(f_out)
        escritor.writerows(linhas)

# Processar os produtos
def processar_produtos(url):
    linhas = baixar_linhas_csv_da_url(url)
    cabecalho, dados = linhas[0], linhas[1:]

    indices = {
        'categoria': cabecalho.index('product_category_name'),
        'peso': cabecalho.index('product_weight_g'),
        'comprimento': cabecalho.index('product_length_cm'),
        'altura': cabecalho.index('product_height_cm'),
        'largura': cabecalho.index('product_width_cm'),
    }

    medias = {
        'peso': calcular_media_coluna(dados, indices['peso']),
        'comprimento': calcular_media_coluna(dados, indices['comprimento']),
        'altura': calcular_media_coluna(dados, indices['altura']),
        'largura': calcular_media_coluna(dados, indices['largura']),
    }

    tabela_corrigida = [cabecalho]
    for linha in dados:
        tabela_corrigida.append(corrigir_linha(linha, indices, medias))

    return tabela_corrigida


# Baixar, tratar e salvar
url_products = 'https://raw.githubusercontent.com/fiesc-junior-prado/mine_projeto_bloco_1/refs/heads/main/olist_products_dataset.csv'
tabela_corrigida = processar_produtos(url_products)
salvar_csv('produtos_processados.csv', tabela_corrigida)

print("Arquivo processado salvo como 'produtos_processados.csv'")
print(tabulate(tabela_corrigida[:11], headers='firstrow', tablefmt='grid'))


Arquivo processado salvo como 'produtos_processados.csv'
+----------------------------------+-------------------------+-----------------------+------------------------------+----------------------+--------------------+---------------------+---------------------+--------------------+
| product_id                       | product_category_name   |   product_name_lenght |   product_description_lenght |   product_photos_qty |   product_weight_g |   product_length_cm |   product_height_cm |   product_width_cm |
+==================================+=========================+=======================+==============================+======================+====================+=====================+=====================+====================+
| 1e9e8ef04dbcff4541ed26657ea517e5 | perfumaria              |                    40 |                          287 |                    1 |                225 |                  16 |                  10 |                 14 |
+----------------------------------